In [ ]:
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

from spleen_qc_seg import config
from spleen_qc_seg.qc.check_dataset import get_nifti_paths


In [ ]:
#Inspect raw file structure

python
raw_dir = config.RAW_DIR
print(raw_dir)
print((raw_dir / "imagesTr").glob("*.nii.gz"))
print((raw_dir / "labelsTr").glob("*.nii.gz"))

images, labels = get_nifti_paths(raw_dir)
len(images), len(labels)


In [ ]:
Look at basic shapes, spacings, intensity ranges for a few cases

python
def summarize_volume(path):
    img = nib.load(str(path))
    data = img.get_fdata()
    return {
        "path": path.name,
        "shape": data.shape,
        "spacing": img.header.get_zooms(),
        "min": float(np.nanmin(data)),
        "max": float(np.nanmax(data)),
    }

summaries = [summarize_volume(p) for p in images[:5]]
summaries


In [ ]:
Plot histograms of image spacings and intensities

python
all_spacings = []
all_mins = []
all_maxs = []

for p in images:
    img = nib.load(str(p))
    data = img.get_fdata()
    all_spacings.append(img.header.get_zooms())
    all_mins.append(np.nanmin(data))
    all_maxs.append(np.nanmax(data))

all_spacings = np.array(all_spacings)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.hist(all_spacings[:, 0], bins=10)
plt.title("Spacing x")

plt.subplot(1, 3, 2)
plt.hist(all_spacings[:, 1], bins=10)
plt.title("Spacing y")

plt.subplot(1, 3, 3)
plt.hist(all_spacings[:, 2], bins=10)
plt.title("Spacing z")

plt.tight_layout()
plt.show()


In [ ]:
Visualize a random case (image + label overlay)

python
import random

idx = random.randint(0, len(images) - 1)
img_path = images[idx]
lbl_path = labels[idx]
print(img_path.name, lbl_path.name)

img = nib.load(str(img_path)).get_fdata()
lbl = nib.load(str(lbl_path)).get_fdata()

slice_idx = img.shape[2] // 2
img_slice = img[:, :, slice_idx]
lbl_slice = lbl[:, :, slice_idx]

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("CT image")
plt.imshow(img_slice.T, cmap="gray", origin="lower")

plt.subplot(1, 2, 2)
plt.title("Label overlay")
plt.imshow(img_slice.T, cmap="gray", origin="lower")
plt.imshow(
    np.ma.masked_where(lbl_slice.T == 0, lbl_slice.T),
    cmap="Reds",
    alpha=0.5,
    origin="lower",
)

plt.tight_layout()
plt.show()


In [ ]:
Hook into your QC functions and inspect the JSON

python
from spleen_qc_seg.qc.check_dataset import check_dataset
import json

check_dataset()  # creates dataset_stats.json

dq_json = config.REPORTS_DIR / "data_quality" / "dataset_stats.json"
dq_stats = json.loads(dq_json.read_text())
list(dq_stats.keys()), len(dq_stats["images"]), len(dq_stats["labels"])